# Watermark Removal with Deep Learning
## U-Net Architecture for Image Inpainting

This notebook implements a watermark removal model using:
- **U-Net architecture** with encoder-decoder structure
- **Skip connections** to preserve spatial information
- **Perceptual loss** for high-quality reconstruction
- **Data augmentation** for robust training

## 1. Setup and Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# GPU Configuration
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✓ GPU available: {gpus}")
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print("⚠ No GPU found, using CPU")

# Configuration
CONFIG = {
    'IMAGE_SIZE': (256, 256),
    'BATCH_SIZE': 16,
    'EPOCHS': 100,
    'LEARNING_RATE': 1e-4,
    'SEED': 42
}

# Set seeds for reproducibility
np.random.seed(CONFIG['SEED'])
tf.random.set_seed(CONFIG['SEED'])

print(f"TensorFlow version: {tf.__version__}")
print(f"Configuration: {CONFIG}")

## 2. Dataset Loading and Exploration

In [ ]:
# Update these paths based on your Kaggle dataset location
# If you uploaded as a dataset, it will be in /kaggle/input/your-dataset-name/
BASE_PATH = '/kaggle/input/watermark-removal-dataset/watermark_removal_dataset'

TRAIN_CSV = os.path.join(BASE_PATH, 'wm-nown', 'train_pairs.csv')
TEST_CSV = os.path.join(BASE_PATH, 'wm-nown', 'test_pairs.csv')
WATERMARKED_DIR = os.path.join(BASE_PATH, 'watermarked')
CLEAN_DIR = os.path.join(BASE_PATH, 'nonwatermarked')

# Load dataset info
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"Training pairs: {len(train_df)}")
print(f"Testing pairs: {len(test_df)}")
print(f"\nTrain CSV columns: {train_df.columns.tolist()}")
print(f"\nFirst few entries:")
print(train_df.head())

In [ ]:
# Visualize sample image pairs
def show_image_pairs(df, num_pairs=3):
    fig, axes = plt.subplots(num_pairs, 2, figsize=(12, 4*num_pairs))
    
    for idx in range(num_pairs):
        row = df.iloc[idx]
        
        # Load images
        wm_img = Image.open(os.path.join(WATERMARKED_DIR, row['watermarked']))
        clean_img = Image.open(os.path.join(CLEAN_DIR, row['clean']))
        
        axes[idx, 0].imshow(wm_img)
        axes[idx, 0].set_title('Watermarked (Input)', fontsize=12)
        axes[idx, 0].axis('off')
        
        axes[idx, 1].imshow(clean_img)
        axes[idx, 1].set_title('Clean (Target)', fontsize=12)
        axes[idx, 1].axis('off')
    
    plt.tight_layout()
    plt.show()

show_image_pairs(train_df, num_pairs=3)

## 3. Data Pipeline with Augmentation

In [ ]:
def load_image_pair(watermarked_path, clean_path, img_size=(256, 256)):
    """Load and preprocess image pair"""
    # Load images
    wm_img = tf.io.read_file(watermarked_path)
    wm_img = tf.image.decode_jpeg(wm_img, channels=3)
    
    clean_img = tf.io.read_file(clean_path)
    clean_img = tf.image.decode_jpeg(clean_img, channels=3)
    
    # Resize
    wm_img = tf.image.resize(wm_img, img_size)
    clean_img = tf.image.resize(clean_img, img_size)
    
    # Normalize to [-1, 1]
    wm_img = (tf.cast(wm_img, tf.float32) / 127.5) - 1.0
    clean_img = (tf.cast(clean_img, tf.float32) / 127.5) - 1.0
    
    return wm_img, clean_img

def augment_image_pair(wm_img, clean_img):
    """Apply random augmentations to both images identically"""
    # Random horizontal flip
    if tf.random.uniform(()) > 0.5:
        wm_img = tf.image.flip_left_right(wm_img)
        clean_img = tf.image.flip_left_right(clean_img)
    
    # Random vertical flip
    if tf.random.uniform(()) > 0.5:
        wm_img = tf.image.flip_up_down(wm_img)
        clean_img = tf.image.flip_up_down(clean_img)
    
    # Random rotation (90, 180, 270 degrees)
    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    wm_img = tf.image.rot90(wm_img, k=k)
    clean_img = tf.image.rot90(clean_img, k=k)
    
    return wm_img, clean_img

def create_dataset(df, watermarked_dir, clean_dir, batch_size, img_size, augment=True, shuffle=True):
    """Create TensorFlow dataset from dataframe"""
    # Get full paths
    wm_paths = [os.path.join(watermarked_dir, f) for f in df['watermarked'].values]
    clean_paths = [os.path.join(clean_dir, f) for f in df['clean'].values]
    
    # Create dataset
    dataset = tf.data.Dataset.from_tensor_slices((wm_paths, clean_paths))
    
    if shuffle:
        dataset = dataset.shuffle(buffer_size=1000, seed=CONFIG['SEED'])
    
    # Load images
    dataset = dataset.map(
        lambda wm, clean: load_image_pair(wm, clean, img_size),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    
    # Apply augmentation
    if augment:
        dataset = dataset.map(
            augment_image_pair,
            num_parallel_calls=tf.data.AUTOTUNE
        )
    
    # Batch and prefetch
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

# Create datasets
train_dataset = create_dataset(
    train_df, WATERMARKED_DIR, CLEAN_DIR,
    CONFIG['BATCH_SIZE'], CONFIG['IMAGE_SIZE'],
    augment=True, shuffle=True
)

val_dataset = create_dataset(
    test_df, WATERMARKED_DIR, CLEAN_DIR,
    CONFIG['BATCH_SIZE'], CONFIG['IMAGE_SIZE'],
    augment=False, shuffle=False
)

print(f"✓ Train dataset: {len(train_df)} images, {len(train_df) // CONFIG['BATCH_SIZE']} batches")
print(f"✓ Val dataset: {len(test_df)} images, {len(test_df) // CONFIG['BATCH_SIZE']} batches")

## 4. U-Net Model Architecture

### Why U-Net?
1. **Encoder-Decoder Structure**: Captures multi-scale features
2. **Skip Connections**: Preserves spatial details from encoder to decoder
3. **Proven for Image Restoration**: Originally designed for medical image segmentation, excellent for pixel-to-pixel tasks

In [ ]:
def conv_block(x, filters, kernel_size=3, activation='relu', batch_norm=True):
    """Convolutional block with optional batch normalization"""
    x = layers.Conv2D(filters, kernel_size, padding='same', kernel_initializer='he_normal')(x)
    if batch_norm:
        x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)
    
    x = layers.Conv2D(filters, kernel_size, padding='same', kernel_initializer='he_normal')(x)
    if batch_norm:
        x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)
    
    return x

def encoder_block(x, filters):
    """Encoder block: conv -> pool"""
    conv = conv_block(x, filters)
    pool = layers.MaxPooling2D(pool_size=(2, 2))(conv)
    return conv, pool

def decoder_block(x, skip_features, filters):
    """Decoder block: upsample -> concat -> conv"""
    x = layers.Conv2DTranspose(filters, (2, 2), strides=(2, 2), padding='same')(x)
    x = layers.Concatenate()([x, skip_features])
    x = conv_block(x, filters)
    return x

def build_unet(input_shape=(256, 256, 3), start_filters=64):
    """Build U-Net model"""
    inputs = layers.Input(shape=input_shape)
    
    # Encoder (Downsampling)
    conv1, pool1 = encoder_block(inputs, start_filters)      # 256x256
    conv2, pool2 = encoder_block(pool1, start_filters * 2)   # 128x128
    conv3, pool3 = encoder_block(pool2, start_filters * 4)   # 64x64
    conv4, pool4 = encoder_block(pool3, start_filters * 8)   # 32x32
    
    # Bottleneck
    bottleneck = conv_block(pool4, start_filters * 16)       # 16x16
    
    # Decoder (Upsampling)
    dec4 = decoder_block(bottleneck, conv4, start_filters * 8)  # 32x32
    dec3 = decoder_block(dec4, conv3, start_filters * 4)        # 64x64
    dec2 = decoder_block(dec3, conv2, start_filters * 2)        # 128x128
    dec1 = decoder_block(dec2, conv1, start_filters)            # 256x256
    
    # Output layer
    outputs = layers.Conv2D(3, (1, 1), activation='tanh', padding='same')(dec1)
    
    model = models.Model(inputs, outputs, name='UNet_WatermarkRemoval')
    return model

# Build model
model = build_unet(
    input_shape=(*CONFIG['IMAGE_SIZE'], 3),
    start_filters=64
)

print("✓ Model built successfully")
model.summary()

## 5. Custom Loss Functions

### Why Multiple Loss Functions?
1. **L1 Loss (MAE)**: Pixel-wise accuracy
2. **Perceptual Loss**: High-level feature similarity using VGG19
3. **SSIM Loss**: Structural similarity for better visual quality

In [ ]:
# Load VGG19 for perceptual loss
vgg = tf.keras.applications.VGG19(include_top=False, weights='imagenet', input_shape=(*CONFIG['IMAGE_SIZE'], 3))
vgg.trainable = False

# Select specific layers for perceptual loss
perceptual_layers = ['block1_conv2', 'block2_conv2', 'block3_conv3', 'block4_conv3']
perceptual_model = models.Model(
    inputs=vgg.input,
    outputs=[vgg.get_layer(name).output for name in perceptual_layers]
)

def perceptual_loss(y_true, y_pred):
    """Perceptual loss using VGG19 features"""
    # Preprocess: convert from [-1, 1] to [0, 255]
    y_true_prep = (y_true + 1.0) * 127.5
    y_pred_prep = (y_pred + 1.0) * 127.5
    
    # Preprocess for VGG
    y_true_prep = tf.keras.applications.vgg19.preprocess_input(y_true_prep)
    y_pred_prep = tf.keras.applications.vgg19.preprocess_input(y_pred_prep)
    
    # Extract features
    true_features = perceptual_model(y_true_prep)
    pred_features = perceptual_model(y_pred_prep)
    
    # Calculate loss across all layers
    loss = 0.0
    for true_feat, pred_feat in zip(true_features, pred_features):
        loss += tf.reduce_mean(tf.abs(true_feat - pred_feat))
    
    return loss / len(perceptual_layers)

def ssim_loss(y_true, y_pred):
    """Structural Similarity Index loss"""
    return 1.0 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=2.0))

def combined_loss(y_true, y_pred):
    """Combined loss: L1 + Perceptual + SSIM"""
    l1 = tf.reduce_mean(tf.abs(y_true - y_pred))
    perceptual = perceptual_loss(y_true, y_pred)
    ssim = ssim_loss(y_true, y_pred)
    
    # Weighted combination
    total_loss = 1.0 * l1 + 0.1 * perceptual + 0.5 * ssim
    return total_loss

print("✓ Loss functions defined")

## 6. Metrics and Callbacks

In [ ]:
# Custom metrics
def psnr_metric(y_true, y_pred):
    """Peak Signal-to-Noise Ratio"""
    return tf.image.psnr(y_true, y_pred, max_val=2.0)

def ssim_metric(y_true, y_pred):
    """Structural Similarity Index"""
    return tf.image.ssim(y_true, y_pred, max_val=2.0)

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG['LEARNING_RATE']),
    loss=combined_loss,
    metrics=['mae', psnr_metric, ssim_metric]
)

# Callbacks
callbacks = [
    ModelCheckpoint(
        'best_watermark_remover.h5',
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

print("✓ Model compiled with callbacks")

## 7. Training

In [ ]:
# Train the model
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=CONFIG['EPOCHS'],
    callbacks=callbacks,
    verbose=1
)

## 8. Training Visualization

In [ ]:
def plot_training_history(history):
    """Plot training metrics"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss
    axes[0, 0].plot(history.history['loss'], label='Train Loss')
    axes[0, 0].plot(history.history['val_loss'], label='Val Loss')
    axes[0, 0].set_title('Loss Over Epochs')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # MAE
    axes[0, 1].plot(history.history['mae'], label='Train MAE')
    axes[0, 1].plot(history.history['val_mae'], label='Val MAE')
    axes[0, 1].set_title('MAE Over Epochs')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('MAE')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # PSNR
    axes[1, 0].plot(history.history['psnr_metric'], label='Train PSNR')
    axes[1, 0].plot(history.history['val_psnr_metric'], label='Val PSNR')
    axes[1, 0].set_title('PSNR Over Epochs (Higher is Better)')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('PSNR (dB)')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # SSIM
    axes[1, 1].plot(history.history['ssim_metric'], label='Train SSIM')
    axes[1, 1].plot(history.history['val_ssim_metric'], label='Val SSIM')
    axes[1, 1].set_title('SSIM Over Epochs (Higher is Better)')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('SSIM')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_training_history(history)

## 9. Evaluation and Results

In [ ]:
# Load best model
best_model = keras.models.load_model(
    'best_watermark_remover.h5',
    custom_objects={
        'combined_loss': combined_loss,
        'psnr_metric': psnr_metric,
        'ssim_metric': ssim_metric
    }
)

# Evaluate on validation set
results = best_model.evaluate(val_dataset, verbose=1)
print("\n=== Validation Results ===")
print(f"Loss: {results[0]:.4f}")
print(f"MAE: {results[1]:.4f}")
print(f"PSNR: {results[2]:.2f} dB")
print(f"SSIM: {results[3]:.4f}")

In [ ]:
def show_predictions(model, dataset, num_samples=5):
    """Visualize model predictions"""
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5*num_samples))
    
    # Get batch from dataset
    for batch_wm, batch_clean in dataset.take(1):
        predictions = model.predict(batch_wm[:num_samples], verbose=0)
        
        for i in range(num_samples):
            # Denormalize images from [-1, 1] to [0, 1]
            wm_img = (batch_wm[i].numpy() + 1.0) / 2.0
            clean_img = (batch_clean[i].numpy() + 1.0) / 2.0
            pred_img = (predictions[i] + 1.0) / 2.0
            
            # Calculate PSNR and SSIM for this image
            psnr = tf.image.psnr(
                batch_clean[i:i+1], 
                predictions[i:i+1], 
                max_val=2.0
            ).numpy()[0]
            ssim = tf.image.ssim(
                batch_clean[i:i+1], 
                predictions[i:i+1], 
                max_val=2.0
            ).numpy()[0]
            
            # Display
            axes[i, 0].imshow(wm_img)
            axes[i, 0].set_title('Input (Watermarked)', fontsize=12)
            axes[i, 0].axis('off')
            
            axes[i, 1].imshow(pred_img)
            axes[i, 1].set_title(f'Predicted (PSNR: {psnr:.2f}dB, SSIM: {ssim:.3f})', fontsize=12)
            axes[i, 1].axis('off')
            
            axes[i, 2].imshow(clean_img)
            axes[i, 2].set_title('Ground Truth (Clean)', fontsize=12)
            axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig('predictions.png', dpi=300, bbox_inches='tight')
    plt.show()

show_predictions(best_model, val_dataset, num_samples=5)

## 10. Export Model for Production

In [ ]:
# Save in multiple formats

# 1. SavedModel format (for TensorFlow Serving)
best_model.save('watermark_remover_savedmodel', save_format='tf')
print("✓ Saved as TensorFlow SavedModel")

# 2. TFLite format (for mobile/edge deployment)
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('watermark_remover.tflite', 'wb') as f:
    f.write(tflite_model)
print("✓ Saved as TFLite model")

# 3. ONNX format (for cross-framework compatibility)
# !pip install tf2onnx
# import tf2onnx
# spec = (tf.TensorSpec((None, 256, 256, 3), tf.float32, name="input"),)
# output_path = "watermark_remover.onnx"
# model_proto, _ = tf2onnx.convert.from_keras(best_model, input_signature=spec, output_path=output_path)
# print("✓ Saved as ONNX model")

print("\n=== Export Complete ===")
print("Models saved:")
print("  - best_watermark_remover.h5 (Keras format)")
print("  - watermark_remover_savedmodel/ (TensorFlow SavedModel)")
print("  - watermark_remover.tflite (TensorFlow Lite)")

## 11. Inference Function

In [ ]:
def remove_watermark(model, image_path, output_path='output_clean.jpg'):
    """Remove watermark from a single image"""
    # Load and preprocess
    img = Image.open(image_path).convert('RGB')
    original_size = img.size
    
    # Resize to model input size
    img_resized = img.resize(CONFIG['IMAGE_SIZE'])
    img_array = np.array(img_resized, dtype=np.float32)
    
    # Normalize to [-1, 1]
    img_normalized = (img_array / 127.5) - 1.0
    img_batch = np.expand_dims(img_normalized, axis=0)
    
    # Predict
    pred = model.predict(img_batch, verbose=0)[0]
    
    # Denormalize to [0, 255]
    pred_img = ((pred + 1.0) * 127.5).astype(np.uint8)
    
    # Resize back to original size
    pred_pil = Image.fromarray(pred_img)
    pred_pil_resized = pred_pil.resize(original_size, Image.LANCZOS)
    
    # Save
    pred_pil_resized.save(output_path, quality=95)
    print(f"✓ Cleaned image saved to: {output_path}")
    
    return pred_pil_resized

# Test inference
test_image_path = os.path.join(WATERMARKED_DIR, test_df.iloc[0]['watermarked'])
result = remove_watermark(best_model, test_image_path, 'test_output.jpg')

# Display
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(Image.open(test_image_path))
axes[0].set_title('Original (Watermarked)')
axes[0].axis('off')

axes[1].imshow(result)
axes[1].set_title('Cleaned')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 12. Model Summary and Next Steps

### What We Built:
- **U-Net architecture** with skip connections for precise reconstruction
- **Multi-component loss**: L1 + Perceptual + SSIM for high-quality outputs
- **Data augmentation** for robust generalization
- **Metrics**: PSNR and SSIM for objective quality measurement

### Why It Works:
1. **Skip Connections**: Preserve fine details from input to output
2. **Perceptual Loss**: Ensures outputs look natural (not just pixel-accurate)
3. **SSIM Loss**: Maintains structural similarity
4. **Augmentation**: Model learns to handle various orientations/flips

### Next Steps:
1. **Fine-tuning**: Adjust loss weights for better quality
2. **Advanced Architectures**: Try Attention U-Net or ResUNet
3. **GAN-based**: Implement adversarial loss for photorealistic results
4. **Post-processing**: Add edge enhancement or color correction
5. **Deploy**: Integrate into web app or API

### Expected Performance:
- **PSNR**: 25-35 dB (higher is better)
- **SSIM**: 0.85-0.95 (closer to 1 is better)
- **Visual Quality**: Should remove most watermarks while preserving image details